# LGT Order Refine Eval / Inference

`lgt_order_refine_train.ipynb`가 저장한 checkpoint들을 나중에 따로 평가하고, best checkpoint로 test submission을 생성한다.

기본값은 `qwen2vl_lgt_order_refine_v1/runs/` 아래 가장 최신 run을 평가한다. 특정 run을 평가하려면 설정 셀의 `REFINE_RUN_ID`를 지정한다.

In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_lgt_order_refine_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")

In [ ]:
# 2) Setup + data unzip
from google.colab import drive
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
warnings.filterwarnings("ignore", message=".*The following generation flags are not valid.*")
transformers_logging.set_verbosity_error()

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

MODEL_REPO_ID = "Qwen/Qwen2-VL-2B-Instruct"
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-2B-Instruct"


def ensure_base_model_path():
    if os.path.exists(os.path.join(DRIVE_MODEL_DIR, "config.json")):
        print("Using cached base model:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR

    if not USE_MODELSCOPE_BASE_MODEL:
        print("Using Hugging Face repo id:", MODEL_REPO_ID)
        return MODEL_REPO_ID

    print("Base model cache not found. Downloading via ModelScope:")
    print(DRIVE_MODEL_DIR)
    from modelscope import snapshot_download as modelscope_snapshot_download

    model_dir = modelscope_snapshot_download(
        MODEL_REPO_ID,
        cache_dir="/content/modelscope_cache",
    )
    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    if not os.path.exists(DRIVE_MODEL_DIR):
        shutil.copytree(model_dir, DRIVE_MODEL_DIR)
    print("Base model cached at:", DRIVE_MODEL_DIR)
    return DRIVE_MODEL_DIR


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
BASELINE_ADAPTER_DIR = (
    "/content/drive/MyDrive/SNU_AI_Challenge/"
    "qwen2vl_lgt_multitask_v1/runs/"
    "20260712_234828/lgt_multitask/checkpoint-3500"
)
OUTPUT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1"


# Set this to a specific run id when needed, for example "20260713_160355".
# If None, the latest run under OUTPUT_ROOT is used.
REFINE_RUN_ID = None


def resolve_run_root(output_root, run_id=None):
    runs_root = os.path.join(output_root, "runs")
    if run_id is not None:
        run_root = os.path.join(runs_root, run_id)
        assert os.path.isdir(run_root), run_root
        return run_root
    candidates = sorted(
        [
            os.path.join(runs_root, name)
            for name in os.listdir(runs_root)
            if os.path.isdir(os.path.join(runs_root, name))
        ]
    )
    if not candidates:
        raise RuntimeError(f"No runs found under {runs_root}")
    return candidates[-1]


RUN_ROOT = resolve_run_root(OUTPUT_ROOT, REFINE_RUN_ID)
RUN_ID = os.path.basename(RUN_ROOT)
OUTPUT_DIR = os.path.join(RUN_ROOT, "lgt_order_refine")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_adapter")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_lgt_order_refine.csv")

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None

RUN_CONFIG_PATH = os.path.join(RUN_ROOT, "run_config.json")
if os.path.exists(RUN_CONFIG_PATH):
    with open(RUN_CONFIG_PATH, "r", encoding="utf-8") as f:
        RUN_CONFIG = json.load(f)
else:
    RUN_CONFIG = {}
TRAIN_TASK_RATIOS = RUN_CONFIG.get("task_ratios")

QUICK_EVAL_ROWS = 50
FULL_EVAL_ROWS = 300
TOP_K_FULL_EVAL = 3

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))
ALPHAS = [0.5, 1.0, 1.5, 2.0]
BETAS = [0.5, 1.0, 1.5, 2.0]
GAMMAS = [0.5, 1.0, 1.5, 2.0]

for path in [EVAL_DIR, BEST_ADAPTER_DIR]:
    os.makedirs(path, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(BASELINE_ADAPTER_DIR, "adapter_config.json")), BASELINE_ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("baseline adapter:", BASELINE_ADAPTER_DIR)
print("output:", OUTPUT_DIR)
print("eval:", EVAL_DIR)

In [ ]:
# 3) Data split and shared helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def format_order(order):
    return "[" + ", ".join(str(int(value)) for value in order) + "]"


def parse_order_prediction(text):
    match = re.fullmatch(r"\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*", str(text))
    if not match:
        return None
    values = [int(value) for value in match.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None


def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)

if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))

In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the beginning of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "last":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image represents the end of the story?\n"
            "Answer only the image number from 1 to 4."
        )
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Compare the temporal relation between scenes and identify the likely first and last scenes.\n"
            "Using these cues, determine the complete chronological order.\n"
            "Output only the final ordered list, such as [1, 2, 3, 4]."
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


class LGTRefineDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class LGTRefineCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            labels[:] = -100
        else:
            labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        for example in batch:
            text = self.processor.apply_chat_template(make_messages(example, include_answer=True), tokenize=False, add_generation_prompt=False)
            texts.append(text)
            images.append([load_rgb(path) for path in example["image_paths"]])
            task_types.append(example["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        encoded["labels"] = torch.stack([self._mask_prompt(row) for row in encoded["input_ids"]])
        encoded["task_type"] = task_types
        return encoded

In [ ]:
# 5) Load processor/quant config + evaluation helpers
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def model_device(active_model):
    return next(active_model.parameters()).device

def digit_token_id(digit):
    ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Digit {digit} tokenized to {ids}")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: digit_token_id(digit) for digit in [1, 2, 3, 4]}


def checkpoint_name(path):
    return os.path.basename(os.path.normpath(path))


def find_checkpoint_dirs(include_initial=True):
    dirs = []
    if include_initial:
        dirs.append(BASELINE_ADAPTER_DIR)
    for name in os.listdir(OUTPUT_DIR):
        path = os.path.join(OUTPUT_DIR, name)
        if name.startswith("checkpoint-") and os.path.exists(os.path.join(path, "adapter_config.json")):
            dirs.append(path)
    final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
    if os.path.exists(os.path.join(final_dir, "adapter_config.json")):
        dirs.append(final_dir)

    def sort_key(path):
        if path == BASELINE_ADAPTER_DIR:
            return -1
        match = re.findall(r"checkpoint-(\d+)", path)
        if match:
            return int(match[-1])
        return 10**9

    return sorted(dict.fromkeys(dirs), key=sort_key)


def model_device(active_model):
    return next(active_model.parameters()).device


EVAL_MODEL = None
LOADED_ADAPTERS = {}


def adapter_name_for(adapter_dir):
    name = checkpoint_name(adapter_dir)
    return re.sub(r"[^0-9a-zA-Z_]+", "_", name)


def sanitize_generation_config(active_model):
    generation_config = active_model.generation_config
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None
    generation_config.num_beams = 1
    return active_model


def load_eval_model(adapter_dir):
    # Load the Qwen2-VL base model once, then switch LoRA adapters in-place.
    global EVAL_MODEL
    adapter_name = adapter_name_for(adapter_dir)

    if EVAL_MODEL is None:
        base = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
            local_files_only=MODEL_LOCAL_FILES_ONLY,
        )
        EVAL_MODEL = PeftModel.from_pretrained(
            base,
            adapter_dir,
            adapter_name=adapter_name,
            is_trainable=False,
        )
        LOADED_ADAPTERS[adapter_dir] = adapter_name
    else:
        if adapter_dir not in LOADED_ADAPTERS:
            EVAL_MODEL.load_adapter(
                adapter_dir,
                adapter_name=adapter_name,
                is_trainable=False,
            )
            LOADED_ADAPTERS[adapter_dir] = adapter_name
        EVAL_MODEL.set_adapter(LOADED_ADAPTERS[adapter_dir])

    sanitize_generation_config(EVAL_MODEL)
    EVAL_MODEL.eval()
    return EVAL_MODEL


def make_eval_example(row, task_type, pair=None):
    answer = [int(value) for value in row.get("Answer_list", [1, 2, 3, 4])]
    image_paths = row_image_paths(row, TRAIN_IMAGE_DIR)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order_to_sequence(answer),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["first_index"] = a - 1
        example["second_index"] = b - 1
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


@torch.no_grad()
def score_digit_candidates(active_model, example, candidates):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    outputs = active_model(**inputs)
    last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
    logits = outputs.logits[0, last_pos]
    token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
    probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
    processor.tokenizer.padding_side = old_padding_side
    return {int(candidate): float(prob) for candidate, prob in zip(candidates, probs)}


@torch.no_grad()
def generate_order(active_model, row, max_new_tokens=16):
    example = make_eval_example(row, "order")
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = generated[:, inputs["input_ids"].shape[1]:]
    output = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    processor.tokenizer.padding_side = old_padding_side
    return parse_order_prediction(output), output


def order_ranks(order):
    return {int(image_number): position for position, image_number in enumerate(order)}


def pair_accuracy_from_orders(pred_order, gold_order):
    pred_ranks = order_ranks(pred_order)
    gold_ranks = order_ranks(gold_order)
    return np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ])


def order_metric_row(pred_order, gold_order):
    if pred_order is None:
        return {"exact_match": 0.0, "pair_accuracy": 0.0, "position_accuracy": 0.0, "valid_output": 0.0}
    return {
        "exact_match": float(pred_order == gold_order),
        "pair_accuracy": float(pair_accuracy_from_orders(pred_order, gold_order)),
        "position_accuracy": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "valid_output": 1.0,
    }

In [ ]:
# 8) Probability cache and structured decoding
def extract_probability_cache(adapter_dir, rows, tag):
    ckpt = checkpoint_name(adapter_dir)
    cache_path = os.path.join(EVAL_DIR, f"{ckpt}_{tag}_task_probability_cache.json")
    if os.path.exists(cache_path):
        print("[SKIP]", cache_path)
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    eval_model = load_eval_model(adapter_dir)
    records = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{ckpt} {tag} probs"):
        answer = [int(value) for value in row["Answer_list"]]
        gold_order = order_to_sequence(answer)
        first_probs = score_digit_candidates(eval_model, make_eval_example(row, "first"), [1, 2, 3, 4])
        last_probs = score_digit_candidates(eval_model, make_eval_example(row, "last"), [1, 2, 3, 4])
        pair_probs = {}
        pair_correct = []
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            probs = score_digit_candidates(eval_model, make_eval_example(row, "pairwise", pair=(a, b)), [1, 2])
            p_a_before_b = probs[1]
            pair_probs[f"{a}>{b}"] = float(p_a_before_b)
            pair_probs[f"{b}>{a}"] = float(1.0 - p_a_before_b)
            pred_first = a if p_a_before_b >= 0.5 else b
            gold_first = a if answer[first_index] < answer[second_index] else b
            pair_correct.append(int(pred_first == gold_first))

        direct_order, direct_text = generate_order(eval_model, row)
        records.append({
            "sample_id": str(row["Id"]),
            "gold_order": gold_order,
            "first_probs": {str(k): v for k, v in first_probs.items()},
            "last_probs": {str(k): v for k, v in last_probs.items()},
            "pair_probs": pair_probs,
            "direct_order": direct_order,
            "direct_text": direct_text,
            "pairwise_accuracy": float(np.mean(pair_correct)),
            "first_accuracy": float(max(first_probs, key=first_probs.get) == gold_order[0]),
            "last_accuracy": float(max(last_probs, key=last_probs.get) == gold_order[-1]),
        })

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()
    return records


def structured_score(sample, order, alpha, beta, gamma):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(sample["pair_probs"][f"{order[i]}>{order[j]}"]) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(sample["first_probs"][str(order[0])]) + eps)
    last_score = math.log(float(sample["last_probs"][str(order[-1])]) + eps)
    return alpha * pair_score + beta * first_score + gamma * last_score


def decode_structured(sample, alpha, beta, gamma):
    return list(max(PERMUTATIONS, key=lambda order: structured_score(sample, order, alpha, beta, gamma)))


def evaluate_decoding(samples, alpha=1.0, beta=1.0, gamma=1.0, mode="structured"):
    rows = []
    for sample in samples:
        gold = [int(value) for value in sample["gold_order"]]
        if mode == "direct":
            pred = sample["direct_order"]
        else:
            pred = decode_structured(sample, alpha, beta, gamma)
        metric = order_metric_row(pred, gold)
        metric.update({
            "sample_id": sample["sample_id"],
            "pred_order": pred,
            "gold_order": gold,
            "pairwise_accuracy": sample["pairwise_accuracy"],
            "first_accuracy": sample["first_accuracy"],
            "last_accuracy": sample["last_accuracy"],
            "first_last_both_correct": float(sample["first_accuracy"] == 1.0 and sample["last_accuracy"] == 1.0),
        })
        rows.append(metric)
    df = pd.DataFrame(rows)
    summary = {
        "exact_match": df["exact_match"].mean(),
        "pair_accuracy": df["pair_accuracy"].mean(),
        "position_accuracy": df["position_accuracy"].mean(),
        "valid_output_rate": df["valid_output"].mean(),
        "mean_pairwise_accuracy": df["pairwise_accuracy"].mean(),
        "mean_first_accuracy": df["first_accuracy"].mean(),
        "mean_last_accuracy": df["last_accuracy"].mean(),
        "first_last_both_correct_rate": df["first_last_both_correct"].mean(),
        "exact_given_first_last_correct": df.loc[df["first_last_both_correct"] == 1.0, "exact_match"].mean() if (df["first_last_both_correct"] == 1.0).any() else np.nan,
    }
    return summary, df


def grid_search(samples, checkpoint, tag):
    path = os.path.join(EVAL_DIR, f"{checkpoint}_{tag}_decoding_weight_search.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    rows = []
    for alpha, beta, gamma in itertools.product(ALPHAS, BETAS, GAMMAS):
        summary, _ = evaluate_decoding(samples, alpha=alpha, beta=beta, gamma=gamma, mode="structured")
        summary.update({"checkpoint": checkpoint, "tag": tag, "alpha": alpha, "beta": beta, "gamma": gamma, "decoding": "pair_first_last"})
        rows.append(summary)
    df = pd.DataFrame(rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
    df.to_csv(path, index=False)
    return df

In [ ]:
# 9) Quick checkpoint evaluation
quick_rows = validation_df.sample(n=min(QUICK_EVAL_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)
checkpoint_dirs = find_checkpoint_dirs(include_initial=True)
print("checkpoints:", [checkpoint_name(path) for path in checkpoint_dirs])

summary_path = os.path.join(EVAL_DIR, "all_checkpoint_metrics_quick.csv")
if os.path.exists(summary_path):
    summary_df = pd.read_csv(summary_path)
    completed = set(summary_df["checkpoint"].astype(str))
    summary_rows = summary_df.to_dict("records")
else:
    completed = set()
    summary_rows = []

for adapter_dir in checkpoint_dirs:
    ckpt = checkpoint_name(adapter_dir)
    if ckpt in completed:
        print("[SKIP]", ckpt)
        continue
    samples = extract_probability_cache(adapter_dir, quick_rows, tag=f"quick{len(quick_rows)}")
    direct_summary, direct_predictions = evaluate_decoding(samples, mode="direct")
    direct_predictions.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_quick_direct_predictions.csv"), index=False)
    direct_summary.update({"checkpoint": ckpt, "tag": f"quick{len(quick_rows)}", "decoding": "direct", "alpha": np.nan, "beta": np.nan, "gamma": np.nan})
    summary_rows.append(direct_summary)

    search = grid_search(samples, ckpt, tag=f"quick{len(quick_rows)}")
    best_structured = search.iloc[0].to_dict()
    summary_rows.append(best_structured)
    pd.DataFrame(summary_rows).to_csv(summary_path, index=False)

summary_df = pd.DataFrame(summary_rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
display(summary_df)
top_checkpoints = summary_df[summary_df["decoding"].eq("pair_first_last")].head(TOP_K_FULL_EVAL)["checkpoint"].tolist()
print("top structured checkpoints:", top_checkpoints)

In [ ]:
# 10) Full validation for top checkpoints
full_rows = validation_df.sample(n=min(FULL_EVAL_ROWS, len(validation_df)), random_state=SEED + 1).reset_index(drop=True)
full_summary_path = os.path.join(EVAL_DIR, "all_checkpoint_metrics_full.csv")
if os.path.exists(full_summary_path):
    full_summary_df = pd.read_csv(full_summary_path)
    completed = set(full_summary_df["checkpoint"].astype(str) + "::" + full_summary_df["decoding"].astype(str))
    full_rows_out = full_summary_df.to_dict("records")
else:
    completed = set()
    full_rows_out = []

name_to_dir = {checkpoint_name(path): path for path in checkpoint_dirs}
for ckpt in top_checkpoints:
    adapter_dir = name_to_dir[ckpt]
    samples = extract_probability_cache(adapter_dir, full_rows, tag=f"full{len(full_rows)}")
    key = ckpt + "::direct"
    if key not in completed:
        direct_summary, direct_predictions = evaluate_decoding(samples, mode="direct")
        direct_predictions.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_full_direct_predictions.csv"), index=False)
        direct_summary.update({"checkpoint": ckpt, "tag": f"full{len(full_rows)}", "decoding": "direct", "alpha": np.nan, "beta": np.nan, "gamma": np.nan})
        full_rows_out.append(direct_summary)
        pd.DataFrame(full_rows_out).to_csv(full_summary_path, index=False)

    key = ckpt + "::pair_first_last"
    if key not in completed:
        search = grid_search(samples, ckpt, tag=f"full{len(full_rows)}")
        best = search.iloc[0].to_dict()
        _, structured_predictions = evaluate_decoding(samples, alpha=best["alpha"], beta=best["beta"], gamma=best["gamma"], mode="structured")
        structured_predictions.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_full_structured_predictions.csv"), index=False)
        full_rows_out.append(best)
        pd.DataFrame(full_rows_out).to_csv(full_summary_path, index=False)

full_summary_df = pd.DataFrame(full_rows_out).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False).reset_index(drop=True)
display(full_summary_df)

best = full_summary_df.iloc[0].to_dict()
best_checkpoint = best["checkpoint"]
best_adapter_dir = name_to_dir[best_checkpoint]
print("BEST:", best)

if os.path.exists(BEST_ADAPTER_DIR) and not os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")):
    shutil.rmtree(BEST_ADAPTER_DIR)
os.makedirs(BEST_ADAPTER_DIR, exist_ok=True)
if not os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")):
    for filename in os.listdir(best_adapter_dir):
        src = os.path.join(best_adapter_dir, filename)
        dst = os.path.join(BEST_ADAPTER_DIR, filename)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    processor.save_pretrained(BEST_ADAPTER_DIR)

best_config = {
    "checkpoint": best_checkpoint,
    "checkpoint_dir": best_adapter_dir,
    "decoding": best["decoding"],
    "alpha": None if pd.isna(best.get("alpha", np.nan)) else float(best["alpha"]),
    "beta": None if pd.isna(best.get("beta", np.nan)) else float(best["beta"]),
    "gamma": None if pd.isna(best.get("gamma", np.nan)) else float(best["gamma"]),
    "task_ratio": TRAIN_TASK_RATIOS,
    "full_exact_match": float(best["exact_match"]),
    "full_pair_accuracy": float(best["pair_accuracy"]),
    "full_position_accuracy": float(best["position_accuracy"]),
}
with open(os.path.join(BEST_ADAPTER_DIR, "best_config.json"), "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)
shutil.copy2(full_summary_path, os.path.join(BEST_ADAPTER_DIR, "all_checkpoint_metrics.csv"))
print("best adapter saved:", BEST_ADAPTER_DIR)

In [ ]:
# 11) Test inference and submission
def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def make_test_example(row, task_type, pair=None):
    image_paths = row_image_paths(row, TEST_IMAGE_DIR)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": [1, 2, 3, 4],
        "order": [1, 2, 3, 4],
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


@torch.no_grad()
def generate_test_order(active_model, row, max_new_tokens=16):
    example = make_test_example(row, "order")
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(active_model.device) if torch.is_tensor(value) else value for key, value in inputs.items()}
    generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = generated[:, inputs["input_ids"].shape[1]:]
    output = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    processor.tokenizer.padding_side = old_padding_side
    return parse_order_prediction(output), output


@torch.no_grad()
def test_digit_probs(active_model, row, task_type, candidates, pair=None):
    example = make_test_example(row, task_type, pair=pair)
    return score_digit_candidates(active_model, example, candidates)


with open(os.path.join(BEST_ADAPTER_DIR, "best_config.json"), "r", encoding="utf-8") as f:
    best_config = json.load(f)

test_model = load_eval_model(BEST_ADAPTER_DIR)
submission_rows = []
test_cache = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="test inference"):
    if best_config["decoding"] == "direct":
        pred_order, text = generate_test_order(test_model, row)
        if pred_order is None:
            pred_order = [1, 2, 3, 4]
    else:
        first_probs = test_digit_probs(test_model, row, "first", [1, 2, 3, 4])
        last_probs = test_digit_probs(test_model, row, "last", [1, 2, 3, 4])
        pair_probs = {}
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            probs = test_digit_probs(test_model, row, "pairwise", [1, 2], pair=(a, b))
            pair_probs[f"{a}>{b}"] = float(probs[1])
            pair_probs[f"{b}>{a}"] = float(1.0 - probs[1])
        sample = {
            "sample_id": str(row["Id"]),
            "first_probs": {str(k): v for k, v in first_probs.items()},
            "last_probs": {str(k): v for k, v in last_probs.items()},
            "pair_probs": pair_probs,
        }
        pred_order = decode_structured(sample, best_config["alpha"], best_config["beta"], best_config["gamma"])
        test_cache.append(sample | {"pred_order": pred_order})

    submission_rows.append({"Id": str(row["Id"]), "Answer": str(sequence_to_answer(pred_order))})

submission = pd.DataFrame(submission_rows)
submission.to_csv(SUBMIT_PATH, index=False)
with open(os.path.join(EVAL_DIR, "test_probability_cache.json"), "w", encoding="utf-8") as f:
    json.dump(test_cache, f, ensure_ascii=False, indent=2)
shutil.copy2(SUBMIT_PATH, os.path.join(BEST_ADAPTER_DIR, "submission.csv"))
display(submission.head())
print("submission saved:", SUBMIT_PATH)

## 산출물

```text
eval/all_checkpoint_metrics_quick.csv
eval/all_checkpoint_metrics_full.csv
eval/*_task_probability_cache.json
eval/*_decoding_weight_search.csv
best_adapter/best_config.json
submission_lgt_order_refine.csv
```